# Modul 3: Architektur eines Orchestrators — Schichten verstehen

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook simulierst du das 4-Schichten-Modell eines Agent-Orchestrators als Python-Klassen.

🎯 **Lernziel:** Die 4 Schichten (Channel → Gateway → Session → Runtime) verstehen und als Code nachbauen.

---

## Das 4-Schichten-Modell

Jeder Orchestrator folgt derselben Grundstruktur — egal ob OpenClaw, LangGraph oder AutoGen:

```
┌─────────────────────────────────┐
│  CHANNEL (Telegram, WhatsApp)   │  ← Input/Output
├─────────────────────────────────┤
│  GATEWAY (Routing, Auth)        │  ← Control Plane
├─────────────────────────────────┤
│  SESSION (State, History)       │  ← Zustandsverwaltung
├─────────────────────────────────┤
│  RUNTIME (LLM, Tools, Loop)     │  ← Execution
└─────────────────────────────────┘
```

**Warum Schichten?** Jede Schicht hat eine klare Verantwortung. Das macht das System testbar, austauschbar und erweiterbar.

| Schicht | Verantwortung | Beispiel |
|---------|--------------|----------|
| Channel | Nachrichten empfangen/senden | Telegram-Bot, WhatsApp, CLI |
| Gateway | Routing, Authentifizierung | Welcher Agent? Erlaubt? |
| Session | Gesprächszustand halten | Chat-History, User-Kontext |
| Runtime | Denken + Handeln | LLM-Aufruf, Tool-Execution |

In [ ]:
# Das 4-Schichten-Modell als Python-Klassen

from datetime import datetime


class Channel:
    """Schicht 1: Empfängt Nachrichten von einer Plattform."""
    
    def __init__(self, name):
        self.name = name  # z.B. 'telegram', 'whatsapp'
    
    def receive(self, text, user_id):
        """Nachricht empfangen und normalisieren."""
        message = {
            'text': text,
            'user_id': user_id,
            'channel': self.name,
            'timestamp': datetime.now().isoformat()
        }
        print(f'📨 [{self.name}] Nachricht von {user_id}: "{text}"')
        return message
    
    def send(self, text, user_id):
        """Antwort an den User senden."""
        print(f'📤 [{self.name}] Antwort an {user_id}: "{text}"')


class Gateway:
    """Schicht 2: Routing und Zugriffskontrolle."""
    
    def __init__(self):
        self.allowed_users = set()
        self.routes = {}  # channel_name -> agent_name
    
    def allow_user(self, user_id):
        self.allowed_users.add(user_id)
    
    def add_route(self, channel_name, agent_name):
        self.routes[channel_name] = agent_name
    
    def process(self, message):
        """Prüfe Berechtigung und route zur richtigen Session."""
        user_id = message['user_id']
        channel = message['channel']
        
        # Authentifizierung
        if user_id not in self.allowed_users:
            print(f'🚫 Gateway: User {user_id} nicht autorisiert!')
            return None
        
        # Routing
        agent = self.routes.get(channel, 'default')
        print(f'🔀 Gateway: Route {channel} → Agent "{agent}"')
        message['agent'] = agent
        return message


class Session:
    """Schicht 3: Gesprächszustand und History."""
    
    def __init__(self, session_id):
        self.session_id = session_id
        self.history = []
    
    def add_message(self, role, content):
        self.history.append({'role': role, 'content': content})
    
    def get_context(self):
        """Kontext für die Runtime zusammenstellen."""
        return {
            'session_id': self.session_id,
            'history': self.history[-10:],  # Letzte 10 Nachrichten
            'turn_count': len(self.history)
        }
    
    def __repr__(self):
        return f'Session({self.session_id}, {len(self.history)} msgs)'


class Runtime:
    """Schicht 4: LLM-Aufruf und Tool-Execution (simuliert)."""
    
    def __init__(self, persona='Du bist ein hilfreicher Assistent.'):
        self.persona = persona
    
    def execute(self, context, user_message):
        """Simuliert einen LLM-Aufruf."""
        turn = context['turn_count'] + 1
        # Einfache regelbasierte Antwort (kein echtes LLM)
        if 'hallo' in user_message.lower():
            response = f'Hallo! Ich bin dein Agent. (Turn {turn})'
        elif 'hilfe' in user_message.lower():
            response = f'Ich kann dir helfen! Was brauchst du? (Turn {turn})'
        elif '?' in user_message:
            response = f'Gute Frage! Lass mich nachdenken... (Turn {turn})'
        else:
            response = f'Verstanden: "{user_message}" (Turn {turn})'
        
        print(f'🤖 Runtime: {response}')
        return response


# === Alles zusammenstecken ===
print('=== Orchestrator Demo ===\n')

# Schichten initialisieren
telegram = Channel('telegram')
gateway = Gateway()
session_store = {}  # session_id -> Session
runtime = Runtime()

# Konfigurieren
gateway.allow_user('jan')
gateway.add_route('telegram', 'planck')

# Nachricht verarbeiten
for text in ['Hallo!', 'Wie geht es dir?', 'Hilfe bitte']:
    print(f'\n--- Neue Nachricht ---')
    
    # 1. Channel empfängt
    msg = telegram.receive(text, 'jan')
    
    # 2. Gateway routet
    msg = gateway.process(msg)
    if msg is None:
        continue
    
    # 3. Session laden/erstellen
    sid = f'{msg["user_id"]}:{msg["agent"]}'
    if sid not in session_store:
        session_store[sid] = Session(sid)
    session = session_store[sid]
    session.add_message('user', text)
    
    # 4. Runtime ausführen
    ctx = session.get_context()
    response = runtime.execute(ctx, text)
    session.add_message('assistant', response)
    
    # Antwort zurücksenden
    telegram.send(response, 'jan')

print(f'\n📊 Session-Status: {session_store}')

In [ ]:
# Session-Isolation demonstrieren
# Zwei User, zwei Sessions — kein Datenleck!

print('=== Session-Isolation Test ===\n')

gateway.allow_user('alice')
gateway.allow_user('bob')

sessions = {}

for user, text in [('alice', 'Mein Geheimnis: 42'), ('bob', 'Was hat Alice gesagt?'), ('alice', 'Was weißt du?')]:
    msg = telegram.receive(text, user)
    msg = gateway.process(msg)
    
    sid = f'{msg["user_id"]}:{msg["agent"]}'
    if sid not in sessions:
        sessions[sid] = Session(sid)
    s = sessions[sid]
    s.add_message('user', text)
    
    ctx = s.get_context()
    # Bob kann Alices History NICHT sehen
    print(f'   Session "{sid}" sieht {len(ctx["history"])} Nachrichten')
    response = runtime.execute(ctx, text)
    s.add_message('assistant', response)
    print()

In [ ]:
# 🎯 ÜBUNG: Erweitere den Gateway um Multi-Agent-Routing
#
# Aufgabe: Der Gateway soll verschiedene Channels an verschiedene Agents routen.
# Telegram → 'planck' (persönlich), WhatsApp → 'work-agent' (beruflich)
# Bonus: Füge eine Blocklist hinzu (bestimmte User blockieren).

class AdvancedGateway(Gateway):
    def __init__(self):
        super().__init__()
        self.blocklist = set()
    
    def block_user(self, user_id):
        # TODO: User zur Blocklist hinzufügen
        pass
    
    def process(self, message):
        # TODO: Erst Blocklist prüfen, dann normales Routing
        # Blockierte User sollen eine Warnung bekommen (return None + print)
        pass

# Test
adv_gw = AdvancedGateway()
adv_gw.allow_user('jan')
adv_gw.allow_user('spammer')
adv_gw.block_user('spammer')  # Soll blockiert werden
adv_gw.add_route('telegram', 'planck')
adv_gw.add_route('whatsapp', 'work-agent')

# Diese Nachricht sollte durchkommen:
msg1 = {'text': 'Hi', 'user_id': 'jan', 'channel': 'telegram', 'timestamp': '...'}
# Diese sollte blockiert werden:
msg2 = {'text': 'Buy crypto!', 'user_id': 'spammer', 'channel': 'telegram', 'timestamp': '...'}

print(adv_gw.process(msg1))  # Sollte funktionieren
print(adv_gw.process(msg2))  # Sollte None zurückgeben

In [ ]:
# ✅ LÖSUNG

class AdvancedGateway(Gateway):
    def __init__(self):
        super().__init__()
        self.blocklist = set()
    
    def block_user(self, user_id):
        """User zur Blocklist hinzufügen."""
        self.blocklist.add(user_id)
        print(f'🚫 User "{user_id}" blockiert.')
    
    def process(self, message):
        """Blocklist prüfen, dann normales Routing."""
        user_id = message['user_id']
        
        # Blocklist hat Vorrang
        if user_id in self.blocklist:
            print(f'🛑 Gateway: User "{user_id}" ist blockiert!')
            return None
        
        # Normales Routing (von der Elternklasse)
        return super().process(message)

# Test
adv_gw = AdvancedGateway()
adv_gw.allow_user('jan')
adv_gw.allow_user('spammer')
adv_gw.block_user('spammer')
adv_gw.add_route('telegram', 'planck')
adv_gw.add_route('whatsapp', 'work-agent')

msg1 = {'text': 'Hi', 'user_id': 'jan', 'channel': 'telegram', 'timestamp': '...'}
msg2 = {'text': 'Buy crypto!', 'user_id': 'spammer', 'channel': 'telegram', 'timestamp': '...'}

print('\n--- Test 1: Erlaubter User ---')
result1 = adv_gw.process(msg1)
print(f'Ergebnis: {result1}')

print('\n--- Test 2: Blockierter User ---')
result2 = adv_gw.process(msg2)
print(f'Ergebnis: {result2}')

# Bonus: WhatsApp-Route testen
print('\n--- Test 3: WhatsApp-Route ---')
msg3 = {'text': 'Meeting morgen?', 'user_id': 'jan', 'channel': 'whatsapp', 'timestamp': '...'}
result3 = adv_gw.process(msg3)
print(f'Ergebnis: {result3}')

## Vergleich: Architektur-Ansätze

| Framework | Architektur-Modell | Session-Handling |
|-----------|-------------------|------------------|
| **OpenClaw** | Channel → Gateway → Session → Runtime | File-basiert, isoliert |
| **LangGraph** | Graph (Nodes + Edges) | State-Machine |
| **CrewAI** | Crew → Tasks → Agents | Shared Memory |
| **AutoGen** | GroupChat → Agents | Conversation-basiert |

---

### ✅ Self-Check
- [ ] Ich kann die 4 Schichten benennen und ihre Verantwortung erklären
- [ ] Ich verstehe Session-Isolation (warum User A nicht User Bs Daten sieht)
- [ ] Ich kann erklären warum Schichten-Architektur besser ist als ein Monolith

**→ Weiter zu [Modul 4: Tool Use & Kontrollierte Autonomie](https://colab.research.google.com/github/planck-lab/agent-systems/blob/main/notebooks/modul4_tool_use.ipynb)**